# 模块3：特征工程全流程教学
---
## 🎯 学习目标
学完本模块你将能够：
- 理解推荐算法比赛中特征工程的核心思路
- 掌握8大类常用特征的构建方法和业务逻辑
- 学会时间窗口特征、交叉特征、业务特征的设计思路
- 掌握特征变换、缺失值处理等后处理方法
- 能够独立完成用户行为类数据集的完整特征工程流程

---
## 📚 知识点讲解：特征工程基础

### 1. 什么是特征工程？
特征工程是将原始数据转换为模型能够理解的输入特征的过程，它决定了模型效果的上限，好的特征工程能够让简单的模型也能取得很好的效果。

在推荐算法比赛中，特征工程通常占整个项目70%以上的工作量，是决定比赛排名的关键因素。

### 2. 推荐场景常用特征分类
| 特征类别 | 说明 | 例子 |
|----------|------|------|
| 用户侧特征 | 描述用户本身的属性和行为习惯 | 用户总浏览量、购买率、活跃度 |
| 商品侧特征 | 描述商品本身的属性和受欢迎程度 | 商品总浏览量、转化率、热度 |
| 交互特征 | 描述用户和商品之间的互动关系 | 用户对该商品的浏览/加购/购买次数 |
| 时间特征 | 描述行为的时间属性和时间规律 | 最近一次行为距今天数、不同时间段的活跃度 |
| 统计特征 | 基于统计指标计算的特征 | 平均值、最大值、最小值、标准差 |
| 转化特征 | 描述各环节转化效率的特征 | 浏览加购率、加购购买率 |
| 趋势特征 | 描述行为变化趋势的特征 | 近期行为量与中期行为量的比值 |
| 业务特征 | 结合业务场景设计的特殊特征 | 双十二加购未买标记、大促权重衰减 |

### 3. 本项目特征工程整体架构
我们采用**双流特征工程架构**：
- 流A：全局用户特征，刻画用户整体的行为习惯和购买力
- 流B：局部交互特征，刻画用户对具体商品的兴趣程度
- 加上时间特征、趋势特征、转化特征、业务特征，总共5大类20+特征

---
## 💻 代码逐行拆解：数据准备与预处理


In [ ]:
# 1. 导入需要的库
import pandas as pd
import numpy as np
from datetime import datetime

In [ ]:
# 2. 加载原始数据
df_user = pd.read_csv(
    "F:/数据分析/project/天池/新人实战/fresh_comp_offline/tianchi_fresh_comp_train_user.csv"
)
df_item = pd.read_csv(
    "F:/数据分析/project/天池/新人实战/fresh_comp_offline/tianchi_fresh_comp_train_item.csv"
)

# 3. 时间字段预处理
df_user["time"] = pd.to_datetime(df_user["time"])
df_user["date"] = df_user["time"].dt.date  # 提取日期部分，方便后续按天筛选
df_user["hour"] = df_user["time"].dt.hour

### 时间窗口划分（滑窗法）
在时序预测问题中，我们需要严格划分特征窗口和标签窗口，避免数据泄露：
- **特征窗口**：11月18日 - 12月17日，共30天数据，用于构建特征
- **标签窗口**：12月18日，1天数据，用于构建标签（用户是否购买）
- **预测目标**：预测12月19日用户是否会购买商品子集的商品


In [ ]:
# 划分时间窗口
feat_start = pd.to_datetime("2014-11-18")
feat_end = pd.to_datetime("2014-12-17")
label_day = pd.to_datetime("2014-12-18")

# 切分特征集和标签集
df_feat = df_user[(df_user["time"] >= feat_start) & (df_user["time"] <= feat_end)].copy()
df_label = df_user[df_user["time"] == label_day].copy()

# 加载商品子集
subset_items = set(df_item["item_id"].unique())
print(f"目标商品子集数量：{len(subset_items)}")

⚠️ **重要提醒：数据泄露问题**
绝对不能使用标签窗口的数据来构建特征，否则会出现数据泄露，导致线下分数虚高，线上成绩一塌糊涂。
时间窗口划分是时序问题的生命线，必须严格遵守。

---
## 💻 代码逐行拆解：标签构建

标签是模型要预测的目标，本项目是二分类问题：
- 正样本(label=1)：用户在12月18日购买了商品子集的商品
- 负样本(label=0)：用户在特征期内和商品有交互，但在12月18日没有购买


In [ ]:
# 正样本：12月18日在子集内购买的用户-商品对
pos = df_label[
    (df_label["item_id"].isin(subset_items)) & (df_label["behavior_type"] == 4)
][["user_id", "item_id"]].drop_duplicates()
pos["label"] = 1

# 负样本：特征期内在子集内有交互，但12月18日没购买的用户-商品对
interact_sub = df_feat[df_feat["item_id"].isin(subset_items)][
    ["user_id", "item_id"]
].drop_duplicates()
neg = interact_sub.merge(pos, on=["user_id", "item_id"], how="left")
neg = neg[neg["label"].isnull()][["user_id", "item_id"]]
neg["label"] = 0

# 合并正负样本，得到基础数据集
base_df = pd.concat([pos, neg]).reset_index(drop=True)
print(f"正负样本数量：\n{base_df['label'].value_counts()}")
print(f"样本总数量：{len(base_df)}")

**负采样说明：**
- 我们没有随机采样负样本，而是选择所有有过交互的负样本，这样更符合真实业务场景
- 可以看到正负样本比例非常不平衡（约1:100），后续建模时需要处理样本不平衡问题


---
## 💻 代码逐行拆解：双流基础特征构建

### 流A：全局用户特征
刻画用户整体的行为习惯，是用户的"画像"特征

In [ ]:
user_global = (
    df_feat.groupby("user_id")
    .agg(
        u_g_view=("behavior_type", lambda x: (x == 1).sum()),  # 用户总浏览次数
        u_g_fav=("behavior_type", lambda x: (x == 2).sum()),   # 用户总收藏次数
        u_g_cart=("behavior_type", lambda x: (x == 3).sum()),  # 用户总加购次数
        u_g_buy=("behavior_type", lambda x: (x == 4).sum()),   # 用户总购买次数
    )
    .reset_index()
)

# 计算用户整体转化率
user_global["u_g_conv_rate"] = user_global["u_g_buy"] / (user_global["u_g_view"] + 1)  # +1避免除以0

print("全局用户特征示例：")
display(user_global.head())

**参数详解：**
- `groupby("user_id")`：按用户ID分组，对每个用户的所有行为进行统计
- `.agg()`：聚合函数，可以同时对多个列做不同的聚合操作
- `lambda x: (x == 1).sum()`：匿名函数，统计behavior_type等于1（浏览）的次数

**业务意义：**
- 这些特征刻画了用户的购买力和消费习惯：购买次数多的用户更容易复购，转化率高的用户购买意愿更强


### 流B：局部交互特征
刻画用户对具体商品的兴趣程度，是用户-商品对的交互特征

In [ ]:
# 只筛选商品子集的交互数据
df_sub = df_feat[df_feat["item_id"].isin(subset_items)]

pair_local = (
    df_sub.groupby(["user_id", "item_id"])
    .agg(
        u_i_view=("behavior_type", lambda x: (x == 1).sum()),  # 用户对该商品的浏览次数
        u_i_fav=("behavior_type", lambda x: (x == 2).sum()),   # 用户对该商品的收藏次数
        u_i_cart=("behavior_type", lambda x: (x == 3).sum()),  # 用户对该商品的加购次数
        u_i_buy=("behavior_type", lambda x: (x == 4).sum()),   # 用户对该商品的购买次数
    )
    .reset_index()
)

print("局部交互特征示例：")
display(pair_local.head())

**业务意义：**
- 这些特征直接刻画了用户对某个具体商品的兴趣：加购次数越多，说明用户购买意愿越强；如果已经购买过，复购概率也会更高
- 这是推荐算法中最有区分度的特征类别


In [ ]:
# 合并所有基础特征
df_merged = base_df.merge(pair_local, on=["user_id", "item_id"], how="left")
df_merged = df_merged.merge(user_global, on="user_id", how="left")
df_merged = df_merged.fillna(0)  # 填充缺失值（没有交互的特征为0）

print(f"基础特征构建完成，当前特征数量：{len(df_merged.columns) - 3}")  # 减去user_id, item_id, label

---
## 💻 代码逐行拆解：时间衰减特征

用户的行为是有时间效应的：最近的行为比很久以前的行为更能反映用户当前的兴趣。
我们使用**指数衰减**来给不同时间的行为赋予不同的权重：
$$weight = e^{-\lambda * days\_diff}$$
- $days\_diff$：行为发生日距离预测日的天数
- $\lambda$：衰减系数，值越大衰减越快，通常取0.05-0.1


In [ ]:
def add_time_weighted_features(df_feat, df_current, subset_items):
    df = df_feat.copy()
    pred_dt = pd.to_datetime("2014-12-18")  # 预测基准日
    
    # 计算每个行为距离预测日的天数差
    df["days_diff"] = (pred_dt - df["time"]).dt.total_seconds() / (3600 * 24)
    
    # 指数衰减：行为越近权重越高
    df["weight"] = np.exp(-0.1 * df["days_diff"])
    
    # 【业务优化】双十二大促期间行为降权，因为大促的购买行为不代表常态兴趣
    start_12 = pd.to_datetime("2014-12-08").date()
    end_12 = pd.to_datetime("2014-12-12").date()
    mask_12 = (df["date"] >= start_12) & (df["date"] <= end_12)
    df.loc[mask_12, "weight"] *= 0.3  # 大促期间权重打3折
    
    # 只计算子集商品的加权特征
    df_sub = df[df["item_id"].isin(subset_items)]
    
    # 计算加权特征
    weighted_feats = (
        df_sub.groupby(["user_id", "item_id"])
        .apply(
            lambda g: pd.Series(
                {
                    "w_view": ((g["behavior_type"] == 1) * g["weight"]).sum(),  # 加权浏览量
                    "w_cart": ((g["behavior_type"] == 3) * g["weight"]).sum(),  # 加权加购量
                    "w_fav": ((g["behavior_type"] == 2) * g["weight"]).sum(),   # 加权收藏量
                }
            )
        )
        .reset_index()
    )
    
    # 合并回主表
    df_current = df_current.merge(weighted_feats, on=["user_id", "item_id"], how="left")
    df_current[["w_view", "w_cart", "w_fav"]] = df_current[["w_view", "w_cart", "w_fav"]].fillna(0)
    
    return df_current

# 执行特征构建
df_merged = add_time_weighted_features(df_feat, df_merged, subset_items)
print("时间衰减特征构建完成")

**业务创新点：大促降权**
- 双十二期间用户的购买行为很多是因为促销刺激，不代表用户的常态兴趣
- 对大促期间的行为降权，能够让模型更关注用户的真实长期兴趣，避免过拟合大促的异常行为


---
## 💻 代码逐行拆解：趋势特征

用户行为的变化趋势比静态的统计量更有预测价值：近期行为增多说明用户兴趣在提升，反之则兴趣在下降。

In [ ]:
def add_trend_features(df_feat, df_current, subset_items):
    # 划分时间窗口
    date_recent_start = pd.to_datetime("2014-12-13").date()  # 近期：最近5天
    date_mid_start = pd.to_datetime("2014-12-06").date()    # 中期：之前7天
    date_mid_end = pd.to_datetime("2014-12-12").date()
    
    df = df_feat[df_feat["item_id"].isin(subset_items)]
    
    # 近期行为统计
    df_recent = df[df["date"] >= date_recent_start]
    recent_stats = df_recent.groupby(["user_id", "item_id"]).size().reset_index(name="cnt_recent")
    
    # 中期行为统计
    df_mid = df[(df["date"] >= date_mid_start) & (df["date"] <= date_mid_end)]
    mid_stats = df_mid.groupby(["user_id", "item_id"]).size().reset_index(name="cnt_mid")
    
    # 合并计算趋势
    trend_df = recent_stats.merge(mid_stats, on=["user_id", "item_id"], how="outer").fillna(0)
    trend_df["ratio_post_vs_mid"] = trend_df["cnt_recent"] / (trend_df["cnt_mid"] + 1)  # 近期/中期比率
    trend_df["slope_action"] = trend_df["cnt_recent"] - trend_df["cnt_mid"]  # 差分斜率
    
    # 合并回主表
    df_current = df_current.merge(
        trend_df[["user_id", "item_id", "ratio_post_vs_mid", "slope_action"]],
        on=["user_id", "item_id"],
        how="left"
    )
    df_current[["ratio_post_vs_mid", "slope_action"]] = df_current[["ratio_post_vs_mid", "slope_action"]].fillna(0)
    
    return df_current

# 执行特征构建
df_merged = add_trend_features(df_feat, df_merged, subset_items)
print("趋势特征构建完成")

**业务意义：**
- `ratio_post_vs_mid` > 1说明用户对该商品的兴趣在上升，购买概率更高
- `slope_action` 越大说明兴趣上升越快
- 这类趋势特征往往能带来很明显的效果提升

---
## 💻 代码逐行拆解：转化漏斗特征

各环节的转化率直接反映了用户的购买意愿和商品的吸引力。

In [ ]:
def add_funnel_features(df_feat, df_current):
    # 用户维度转化率
    user_stats = (
        df_feat.groupby("user_id")
        .agg(
            total_view=("behavior_type", lambda x: (x == 1).sum()),
            total_cart=("behavior_type", lambda x: (x == 3).sum()),
            total_buy=("behavior_type", lambda x: (x == 4).sum()),
        )
        .reset_index()
    )
    user_stats["u_view2cart_rate"] = user_stats["total_cart"] / (user_stats["total_view"] + 1)  # 浏览加购率
    user_stats["u_cart2buy_rate"] = user_stats["total_buy"] / (user_stats["total_cart"] + 1)    # 加购购买率
    
    # 商品维度转化率
    item_stats = (
        df_feat.groupby("item_id")
        .agg(
            i_total_view=("behavior_type", lambda x: (x == 1).sum()),
            i_total_buy=("behavior_type", lambda x: (x == 4).sum()),
        )
        .reset_index()
    )
    item_stats["i_conv_rate"] = item_stats["i_total_buy"] / (item_stats["i_total_view"] + 1)  # 商品转化率
    
    # 合并回主表
    df_current = df_current.merge(
        user_stats[["user_id", "u_view2cart_rate", "u_cart2buy_rate"]],
        on="user_id",
        how="left"
    )
    df_current = df_current.merge(
        item_stats[["item_id", "i_conv_rate"]],
        on="item_id",
        how="left"
    )
    
    df_current[["u_view2cart_rate", "u_cart2buy_rate", "i_conv_rate"]] = df_current[
        ["u_view2cart_rate", "u_cart2buy_rate", "i_conv_rate"]
    ].fillna(0)
    
    return df_current

# 执行特征构建
df_merged = add_funnel_features(df_feat, df_merged)
print("漏斗特征构建完成")

**业务意义：**
- 加购购买率高的用户，一旦加购就很可能购买，这类用户是我们重点关注的对象
- 商品转化率高说明商品本身很受欢迎，更容易被购买


---
## 💻 代码逐行拆解：业务特殊特征

结合业务场景设计的特殊特征，往往是比赛提分的关键。

In [ ]:
def add_advanced_features(df_feat, df_current, subset_items):
    # 特征1：双12加购但未买标记（强负特征）
    # 双12加购了但当时没买，说明用户对价格敏感，大促过后更不可能买
    date_1212 = pd.to_datetime("2014-12-12").date()
    df_1212 = df_feat[df_feat["date"] == date_1212]
    
    cart_1212 = df_1212[df_1212["behavior_type"] == 3][["user_id", "item_id"]].drop_duplicates()
    buy_1212 = df_1212[df_1212["behavior_type"] == 4][["user_id", "item_id"]].drop_duplicates()
    
    flag_df = cart_1212.merge(buy_1212, on=["user_id", "item_id"], how="left", indicator=True)
    flag_df["flag_cart1212_no_buy"] = (flag_df["_merge"] == "left_only").astype(int)
    flag_df = flag_df[["user_id", "item_id", "flag_cart1212_no_buy"]]
    
    df_current = df_current.merge(flag_df, on=["user_id", "item_id"], how="left")
    df_current["flag_cart1212_no_buy"] = df_current["flag_cart1212_no_buy"].fillna(0)
    
    # 特征2：交叉特征：用户购买力 * 商品热度
    if "u_g_buy" in df_current.columns and "i_total_view" in df_current.columns:
        df_current["cross_u_buy_i_view"] = df_current["u_g_buy"] * df_current["i_total_view"]
    else:
        df_current["cross_u_buy_i_view"] = 0
    
    return df_current

# 执行特征构建
df_merged = add_advanced_features(df_feat, df_merged, subset_items)
print("高级业务特征构建完成")

---
## 💻 代码逐行拆解：特征后处理

特征构建完成后，还需要做一些后处理，让特征更适合模型学习。

In [ ]:
# 1. 计数类特征做Log变换
# 计数类特征通常是长尾分布，Log变换可以压缩极值，让分布更接近正态
count_cols = [
    "u_g_view", "u_g_fav", "u_g_cart", "u_g_buy",
    "u_i_view", "u_i_fav", "u_i_cart", "u_i_buy",
    "w_view", "w_cart", "w_fav",
    "cnt_recent", "cnt_mid",
    "i_total_view", "i_total_buy",
]

for col in count_cols:
    if col in df_merged.columns:
        df_merged[col] = np.log1p(df_merged[col])  # log1p = log(1+x)，避免log(0)错误

# 2. 比率类特征截断
# 转化率理论上在[0,1]之间，截断可以避免除以0导致的异常值
rate_cols = [
    "u_g_conv_rate", "u_view2cart_rate", "u_cart2buy_rate",
    "i_conv_rate", "ratio_post_vs_mid",
]

for col in rate_cols:
    if col in df_merged.columns:
        df_merged[col] = df_merged[col].clip(0, 1)

# 3. 最终缺失值填充
df_merged = df_merged.fillna(0)

# 4. 分离特征和标签
exclude_cols = ["user_id", "item_id", "label"]
feature_cols = [c for c in df_merged.columns if c not in exclude_cols]

X = df_merged[feature_cols]
y = df_merged["label"]

print(f"✅ 特征工程全部完成！")
print(f"最终特征矩阵形状：{X.shape}")
print(f"特征总数：{len(feature_cols)}")
print(f"特征列表：{feature_cols}")

**为什么要做Log变换？**
- 计数类特征（比如浏览次数）通常是长尾分布：大部分用户浏览次数很少，少数用户浏览次数很多
- 树模型对这种分布的特征学习效果不好，Log变换可以压缩极端值，让特征分布更均匀
- `log1p`而不是`log`是为了避免x=0时出现负无穷


---
## 🎯 实战例题

请你独立完成以下练习：
1. 新增一个特征：用户最后一次对该商品的行为距离预测日的天数
2. 新增用户活跃度特征：用户30天内的活跃天数
3. 新增类目维度特征：用户对该商品所属类目的总浏览、加购、购买次数
4. 尝试设计一个新的业务特征，并说明它的业务意义


---
## ✅ 小测验

### 选择题
1. 以下哪种特征不属于用户-商品交互特征？
   A. 用户对该商品的浏览次数
   B. 用户总购买次数
   C. 用户对该商品的加购次数
   D. 用户对该商品的购买次数

2. 时间衰减特征的主要作用是？
   A. 让更早的行为权重更高
   B. 让更近的行为权重更高
   C. 让所有行为权重一样
   D. 降低所有行为的权重

3. 计数类特征做Log变换的原因是？
   A. 让特征值变大
   B. 让特征值变小
   C. 压缩长尾分布，让特征更适合模型学习
   D. 没有特殊原因，行业惯例

### 简答题
1. 什么是数据泄露？在时序预测问题中如何避免数据泄露？
2. 我们设计了"双十二加购但未买"这个特征，请说明它为什么是强负特征？


---
## 📝 重点总结

本模块核心知识点：
1. **特征工程整体架构**：双流架构（用户全局特征 + 用户-商品交互特征）+ 多维度补充特征
2. **8大类特征构建方法**：
   - 基础统计特征：count、sum、avg等聚合统计
   - 时间衰减特征：指数衰减，近期行为权重更高
   - 时间窗口特征：不同时间窗口的统计对比
   - 趋势特征：行为变化的比率和斜率
   - 转化特征：各环节转化率
   - 交叉特征：不同特征的组合
   - 业务特征：结合业务场景的特殊设计
   - 类别特征：用户/商品/类目等ID特征（本项目未使用，适合深度学习模型）
3. **特征后处理**：Log变换、缺失值填充、异常值截断
4. **比赛技巧**：业务特征往往是提分关键，要深入理解业务场景设计特征

下一个模块我们将学习如何使用LightGBM模型对这些特征进行训练，预测用户的复购概率。